# SprintBoard RL-Only Training (GRPO)

This notebook trains a SprintBoard policy using **RL only** (no SFT phase), and enforces a success gate:
**trained mean reward must beat baseline mean reward**.

Pipeline:
1. Setup + install
2. Build prompt / rollout / reward helpers
3. Evaluate baseline
4. Run GRPO (with retry rounds if needed)
5. Save plots + summary JSON

In [ ]:
import os, pathlib

REPO_DIR = pathlib.Path('sprint_planning_env_repo')
if not REPO_DIR.exists() and not pathlib.Path('server').exists():
    !git clone -q https://huggingface.co/spaces/vikramsrini/sprint_planning_env {REPO_DIR}
if REPO_DIR.exists():
    os.chdir(REPO_DIR)

!pip install -q -U pip
!pip install -q -r requirements-train.txt
!pip install -q -e .
print('cwd:', os.getcwd())

In [ ]:
import json, random, re, time
from pathlib import Path
from statistics import mean
from typing import List, Dict, Any

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

from sprint_planning_env.models import SprintAction
from sprint_planning_env.server.environment import SprintBoardEnvironment
from sprint_planning_env.server.tasks import TASK_REGISTRY, list_task_ids

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
TASK_IDS = list_task_ids()
MAX_NEW_TOKENS = 512
MAX_PROMPT_TOK = 1024
OUT_DIR = Path('./sprintboard_qwen_lora_rl_only')
ASSETS_DIR = Path('./assets'); ASSETS_DIR.mkdir(exist_ok=True)

random.seed(0); np.random.seed(0); torch.manual_seed(0)

VALID_VERBS = {
    'LIST_BACKLOG','VIEW_STORY','CHECK_DEPS','VIEW_TEAM','VIEW_VELOCITY',
    'VIEW_SPRINT','VIEW_BUGS','VIEW_EPIC','SEARCH_BACKLOG','ESTIMATE',
    'ASSIGN','UNASSIGN','ADD_TO_SPRINT','REMOVE_FROM_SPRINT','SET_PRIORITY',
    'FLAG_RISK','DECOMPOSE','FINALIZE_SPRINT'
}
VERB_LIST = sorted(list(VALID_VERBS))

SYSTEM_PROMPT = '''You are an expert Agile Scrum Master operating SprintBoard.
Given a sprint-planning scenario, produce an ordered PLAN of commands that
investigates the board and fixes the issues, then finalises the sprint.
Output one command per line, no markdown, no commentary.
Always end with FINALIZE_SPRINT.'''

def build_user_text(task_id: str) -> str:
    task = TASK_REGISTRY[task_id]
    return (
        f'TASK_ID: {task_id} (difficulty: {task["difficulty"]})\\n'
        f'SCENARIO ALERT: {task["alert"]}\\n\\n'
        'Produce the full plan now (one command per line):'
    )

def build_chat_messages(task_id: str) -> list:
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_text(task_id)},
    ]

_NUMBERED_LINE = re.compile(r"^\s*(?:\d+[\.)]|[-*])\s*(.+)$")


def normalize_completion_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, dict):
        for key in ("content", "text", "completion"):
            val = completion.get(key)
            if val:
                return str(val)
        return str(completion)
    if isinstance(completion, list):
        if not completion:
            return ""
        first = completion[0]
        return normalize_completion_text(first)
    return str(completion)


def parse_plan(text: Any, max_cmds: int = 20) -> List[str]:
    cmds: List[str] = []
    text = normalize_completion_text(text)
    for raw in (text or "").splitlines():
        line = raw.strip().strip("`").strip()
        if not line:
            continue
        m = _NUMBERED_LINE.match(line)
        candidate = (m.group(1) if m else line).strip().strip("`")
        if not candidate:
            continue
        verb = candidate.split()[0].upper()
        if verb in VALID_VERBS:
            cmds.append(candidate)
        if len(cmds) >= max_cmds:
            break
    return cmds

def rollout_score(task_id: str, plan: List[str]) -> Dict[str, float]:
    env = SprintBoardEnvironment()
    env.reset(task_id=task_id, seed=0)
    grader = 0.0
    steps = 0
    for cmd in plan or ['FINALIZE_SPRINT']:
        obs = env.step(SprintAction(command=cmd))
        steps += 1
        g = obs.metadata.get('grader_score') if obs.metadata else None
        if g is not None:
            grader = float(g)
        if obs.done:
            break
    if not env.board.is_finalized:
        obs = env.step(SprintAction(command='FINALIZE_SPRINT'))
        g = obs.metadata.get('grader_score') if obs.metadata else None
        if g is not None:
            grader = float(g)
        steps += 1
    return {'reward': float(np.clip(grader, 0.0, 1.0)), 'steps': steps}

print(f'Tasks={len(TASK_IDS)} | model={MODEL_ID}')

In [ ]:
print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map='auto',
)
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

def build_input_text(task_id: str) -> str:
    return tokenizer.apply_chat_template(
        build_chat_messages(task_id), tokenize=False, add_generation_prompt=True
    )

dataset = Dataset.from_dict({'prompt': [build_input_text(tid) for tid in TASK_IDS]})

@torch.inference_mode()
def policy_rollout(task_id: str, max_new_tokens: int = MAX_NEW_TOKENS) -> Dict[str, Any]:
    was_training = model.training
    model.eval()
    try:
        text = build_input_text(task_id)
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_PROMPT_TOK).to(model.device)
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    finally:
        if was_training:
            model.train()
    plan = parse_plan(completion)
    rec = rollout_score(task_id, plan)
    rec['plan_len'] = len(plan)
    return rec

def random_rollout(task_id: str, n_steps: int = 8) -> Dict[str, Any]:
    plan = []
    for _ in range(n_steps-1):
        v = random.choice(VERB_LIST)
        if v in ('VIEW_STORY','CHECK_DEPS','VIEW_EPIC','ESTIMATE','ASSIGN','UNASSIGN','ADD_TO_SPRINT','REMOVE_FROM_SPRINT','SET_PRIORITY','FLAG_RISK','DECOMPOSE'):
            v = f"{v} 101"
        plan.append(v)
    plan.append('FINALIZE_SPRINT')
    rec = rollout_score(task_id, plan)
    rec['plan_len'] = len(plan)
    return rec

def evaluate(label: str, fn) -> Dict[str, Dict[str, Any]]:
    out = {}
    print(f'\\n=== {label} ===')
    for tid in TASK_IDS:
        r = fn(tid)
        out[tid] = r
        print(f"{tid:<8} reward={r['reward']:.3f} steps={r['steps']:>2}")
    print(f"{label} mean={mean(v['reward'] for v in out.values()):.4f}")
    return out

baseline_results = evaluate('BASELINE_RANDOM', random_rollout)
baseline_mean = mean(v['reward'] for v in baseline_results.values())
print('baseline_mean =', round(baseline_mean, 4))

In [ ]:
def extract_task_id_from_prompt(prompt: Any) -> str:
    if isinstance(prompt, list):
        prompt = prompt[-1].get("content", "") if prompt and isinstance(prompt[-1], dict) else str(prompt)
    elif isinstance(prompt, dict):
        prompt = prompt.get("content", str(prompt))
    m = re.search(r"TASK_ID:\\s*(task_\\d+)", str(prompt))
    return m.group(1) if m else "task_1"


def trajectory_reward(prompts, completions, **kwargs):
    rewards = []
    for p, c in zip(prompts, completions):
        tid = extract_task_id_from_prompt(p)
        plan = parse_plan(c)
        rewards.append(rollout_score(tid, plan)["reward"])
    return rewards


def format_reward(prompts, completions, **kwargs):
    vals = []
    for c in completions:
        plan = parse_plan(c)
        s = 0.0
        if len(plan) >= 1:
            s += 0.15
        if len(plan) >= 3:
            s += 0.10
        if plan and plan[-1].split()[0].upper() == "FINALIZE_SPRINT":
            s += 0.10
        vals.append(s)
    return vals

def run_grpo_round(epochs: int, lr: float):
    cfg = GRPOConfig(
        output_dir=str(OUT_DIR),
        learning_rate=lr,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_generations=4,
        max_completion_length=MAX_NEW_TOKENS,
        max_prompt_length=MAX_PROMPT_TOK,
        num_train_epochs=epochs,
        logging_steps=1,
        logging_first_step=True,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        report_to='none',
        save_strategy='no',
        dataloader_num_workers=0,
        remove_unused_columns=False,
        beta=0.0,
        seed=42,
        temperature=0.7,
    )
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[trajectory_reward, format_reward],
        args=cfg,
        train_dataset=dataset,
    )
    trainer.train()

trained_results = None
trained_mean = -1.0
for round_idx, (epochs, lr) in enumerate([(2, 5e-7), (3, 7e-7), (4, 1e-6)], start=1):
    print(f'\\n=== GRPO ROUND {round_idx}: epochs={epochs}, lr={lr} ===')
    run_grpo_round(epochs=epochs, lr=lr)
    trained_results = evaluate(f'TRAINED_ROUND_{round_idx}', policy_rollout)
    trained_mean = mean(v['reward'] for v in trained_results.values())
    print(f'round {round_idx} trained_mean={trained_mean:.4f} baseline_mean={baseline_mean:.4f}')
    if trained_mean > baseline_mean:
        print('Success: trained model beat baseline.')
        break

assert trained_mean > baseline_mean, (
    f'RL-only training did not beat baseline. trained={trained_mean:.4f}, baseline={baseline_mean:.4f}'
)

task_ids = list(trained_results.keys())
base_vals = [baseline_results[t]['reward'] for t in task_ids]
train_vals = [trained_results[t]['reward'] for t in task_ids]

fig, ax = plt.subplots(figsize=(10.8, 4.6))
x = np.arange(len(task_ids)); w = 0.36
ax.bar(x - w/2, base_vals, w, label=f'Baseline ({mean(base_vals):.3f})', color='#94a3b8')
ax.bar(x + w/2, train_vals, w, label=f'RL-only trained ({mean(train_vals):.3f})', color='#7c3aed')
ax.set_xticks(x); ax.set_xticklabels([t.replace('task_', 'T') for t in task_ids])
ax.set_ylim(0, 1.05)
ax.set_xlabel('Task'); ax.set_ylabel('Grader score')
ax.set_title('SprintBoard: Baseline vs RL-only GRPO')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(ASSETS_DIR / 'rl_only_before_after_per_task.png', dpi=180)
plt.show()

summary = {
    'model': MODEL_ID,
    'method': 'RL-only (GRPO, no SFT)',
    'baseline_mean': round(mean(base_vals), 4),
    'trained_mean': round(mean(train_vals), 4),
    'delta': round(mean(train_vals) - mean(base_vals), 4),
    'per_task': {
        t: {'baseline': round(b, 4), 'trained': round(tr, 4)}
        for t, b, tr in zip(task_ids, base_vals, train_vals)
    }
}
(ASSETS_DIR / 'training_summary_rl_only.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('Saved:', ASSETS_DIR / 'rl_only_before_after_per_task.png')
print('Saved:', ASSETS_DIR / 'training_summary_rl_only.json')